## Proof of Concept

### **Attribute Inference Attacks on ML Models**

#### Research Question
Can we infer hidden/sensitive attributes from trained ML models by exploiting prediction confidence scores?

#### Experimental Setup
- **Selected Datasets**:
  - [Insurance](./../data/insurance.csv) - Healthcare charges prediction
  - [Personality](./../data/personality.csv) - Introvert/Extrovert classification
- **Model Types**:
  - XGBoost
  - Multi-Layer Perceptron
- **Fitting Scenarios**:
  - Underfitted
  - Optimal
  - Overfitted
- **Attack Strategy**:
  1. Test all possible values for each hidden feature
  2. Select value with highest prediction confidence

#### Key Hypotheses
1. Overfitted models leak more information than well-fitted models
2. Attacks succeed better on training data than test data
3. Different model architectures have different privacy vulnerabilities
4. Different datasets have varying vulnerability levels based on feature correlations

### Setup

#### Required Modules

In [123]:
import os
import json
import pandas as pd
import numpy as np
from typing import (
    List, Tuple, Dict, Union, Any)

# Prepare Data
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, LabelEncoder)
from sklearn.model_selection import train_test_split

# Train Models
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#### Load Configuration

In [130]:
RANDOM_STATE = 42
DATA_DIRECTORY = os.path.join('..', 'data')
MODEL_CLASS = {
    'XGBoost': XGBClassifier,
    'Decision Tree': DecisionTreeClassifier,
    'Random Forest': RandomForestClassifier,
    'Multi-Layer Perceptron': MLPClassifier}

# Data configuration JSON
data_config = os.path.join(
    DATA_DIRECTORY,
    'data_settings.json')
with open(data_config, 'r') as file:
    DATA_CONFIG = json.load(file)

# Model configuration JSON
model_config = os.path.join(
    '..',
    'model_settings.json')
with open(model_config, 'r') as file:
    MODEL_CONFIG = json.load(file)

### Prepare Data

Load _Health Insurance_ data:

In [106]:
config_ins = DATA_CONFIG['insurance']

df_ins = pd.read_csv(
    os.path.join(
        DATA_DIRECTORY,
        config_ins['filename']))
        
df_ins.head()

,AGE,SEX,SMOKER,REGION,BMI_CATEGORY,HAS_CHILDREN,LABEL
0,19,female,yes,southwest,2_OVERWEIGHT,False,Q4
1,18,male,no,southeast,3_OBESITY,True,Q1
2,28,male,no,southeast,3_OBESITY,True,Q2
3,33,male,no,northwest,1_HEALTHY,False,Q5
4,32,male,no,northwest,2_OVERWEIGHT,False,Q1


Separate features and target:

In [107]:
def separate_features(
        data_frame: pd.DataFrame,
        target_column: str
    ) -> Tuple[pd.DataFrame, pd.Series]:
    """Return features X and target y separately."""
    X = data_frame.drop(target_column, axis=1)
    y = data_frame[target_column]
    return X, y


target_ins = config_ins['target']
X_ins, y_raw_ins = separate_features(
    df_ins,
    target_ins)

Encode target column:

In [108]:
def encode_target(
        y_raw: pd.Series
    ) -> np.ndarray:
    """Returns encoded ndarray of target series."""
    is_object = y_raw.dtype == 'object'
    is_categorical = isinstance(y_raw, pd.CategoricalDtype)
    if is_object or is_categorical:
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y_raw)
    else:
        y = y_raw.values
    return y


y_ins = encode_target(y_raw_ins)

Split data into train and test sets:

In [109]:
X_trn_ins, X_tst_ins, y_trn_ins, y_tst_ins = train_test_split(
    X_ins, y_ins,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_ins)

Calculate _baseline_ values for each categorical feature:

In [110]:
def calculate_baselines(
        data_frame: pd.DataFrame,
        categorical_columns: List[str]
    ) -> Dict[str, Union[int, float]]:
    """Return baseline values for attack attempts."""
    baselines = {}
    for column in categorical_columns:
        class_probs = data_frame[column].value_counts(normalize=True)
        baselines[column] = {
            'random': float((class_probs ** 2).sum()),
            'majority': float(class_probs.max()),
            'cardinality': data_frame[column].nunique()}
    return baselines


baselines_ins = calculate_baselines(
    df_ins,
    config_ins['categorical_features'])

Pre-processing pipeline:

In [111]:
def preprocessing_pipeline(
        categorical_columns: List[str],
        numerical_columns: List[str]
    ) -> ColumnTransformer:
    """Returns pre-processing pipeline for given features."""
    return ColumnTransformer(
        transformers=[
            (
                'cat',
                OneHotEncoder(handle_unknown='ignore'),
                categorical_columns),
            (
                'num',
                StandardScaler(),
                numerical_columns)])


preprocessor_ins = preprocessing_pipeline(
    config_ins['categorical_features'],
    config_ins['numerical_features'])

### Train Models

Method to train a model's variants:

In [ ]:
def train_variants(
        model_name: str,
        X_trn: pd.DataFrame,
        y_trn: np.ndarray,
        data_preprocessor: ColumnTransformer,
        model_config: Dict[str, Any]
    ):
    """Return specified model trained with given data."""
    variants = {}
    random_seed = {'random_state': RANDOM_STATE}

    # Underfitted model
    underfit_model = MODEL_CLASS[model_name](
        **random_seed,
        **model_config['base_params'],
        **model_config['underfit_params'])
    underfit_pipeline = Pipeline([
        ('preprocessor', data_preprocessor),
        ('model', underfit_model)])
    underfit_pipeline.fit(X_trn, y_trn)
    variants['Underfit'] = underfit_pipeline

    # Optimal model
    base_model = MODEL_CLASS[model_name](
        **random_seed,
        **model_config['base_params'])
    base_pipeline = Pipeline([
        ('preprocessor', data_preprocessor),
        ('model', base_model)])

    grid_search = GridSearchCV(
        estimator=base_pipeline,
        param_grid=model_config['grid_search_params'],
        cv=3, n_jobs=-1, verbose=0)
    grid_search.fit(X_trn, y_trn)
    variants['Optimal'] = grid_search.best_estimator_

    # Overfitted model
    overfit_model = MODEL_CLASS[model_name](
        **random_seed,
        **model_config['base_params'],
        **model_config['overfit_params'])
    overfit_pipeline = Pipeline([
        ('preprocessor', data_preprocessor),
        ('model', overfit_model)
    ])
    overfit_pipeline.fit(X_trn, y_trn)
    variants['Overfit'] = overfit_pipeline

    return variants


warnings.filterwarnings("ignore", category=ConvergenceWarning)

trained_variants = {}
for model_name, model_config in MODEL_CONFIG.items():
    trained_variants[model_name] = train_variants(
        model_name,
        X_trn_ins, y_trn_ins,
        preprocessor_ins,
        model_config)

### Inference Attack

Method that implements the attribute inference attack from the [Fredrikson et al. paper](https://www.usenix.org/system/files/conference/usenixsecurity14/sec14-paper-fredrikson-privacy.pdf):

>_"The algorithm simply completes the target feature vector with each of the possible values for x1, and then computes a weighted probability estimate that this is the correct value."_

In [113]:
def attribute_inference(
        model_variant: Pipeline,
        X_data: pd.DataFrame,
        feature_column: str,
        feture_values: List[Any]
    ) -> pd.DataFrame:
    """
    Performs attribute inference attack on a single feature.
    """
    mappings = []
    
    for _, row in X_data.iterrows():
        victim_row = row.copy()
        true_value = victim_row[feature_column]
        
        # Generate hypothetical records (one per possible value)
        attack_rows = []
        for value in feature_values:
            hypothetical_row = victim_row.copy()
            hypothetical_row[feature_column] = value
            # hypothetical_row['guess_value'] = value
            attack_rows.append(hypothetical_row)
        
        # attack_df = pd.DataFrame(attack_rows)
        # X_attack = attack_df.drop('guess_value', axis=1)
        X_attack = pd.DataFrame(attack_rows)
        
        # Get prediction probabilities
        probabilities = model_variant.predict_proba(X_attack)
        confidences = np.max(probabilities, axis=1)
        
        # Select guess with highest confidence
        best_index = np.argmax(confidences)
        # inferred_value = attack_df.iloc[best_guess_idx]['guess_value']
        inferred_value = X_attack.iloc[best_index][feature_column]
        
        mappings.append({
            'true_value': true_value,
            'inferred_value': inferred_value,
            'is_correct': true_value == inferred_value
        })
    
    return pd.DataFrame(mappings)

Attempt to guess the correct value for categorical features:

In [115]:
dataset_name = 'Insurance'
attack_results = []

for feature in config_ins['categorical_features']:
    feature_values = X_trn_ins[feature].unique()
    feature_baseline = baselines_ins[feature]

    for model_name in MODEL_CONFIG:
        for variant_name in trained_variants[model_name]:
            model_variant = trained_variants[model_name][variant_name]

            attack_results_trn = attribute_inference(
                model_variant, X_trn_ins,
                feature, feature_values)
            attack_accuracy_trn = attack_results_trn['is_correct'].mean()
            attack_results.append({
                'Dataset': dataset_name,
                'Feature': feature,
                'Model': model_name,
                'Variant': variant_name,
                'Split': 'Train',
                'Attack': attack_accuracy_trn,
                'Random': baselines_ins[feature]['random'],
                'Majority': baselines_ins[feature]['majority']})

            print(X_tst_ins)
            attack_results_tst = attribute_inference(
                model_variant, X_tst_ins,
                feature, feature_values)
            attack_accuracy_tst = attack_results_tst['is_correct'].mean()
            attack_results.append({
                'Dataset': dataset_name,
                'Feature': feature,
                'Model': model_name,
                'Variant': variant_name,
                'Split': 'Test',
                'Attack': attack_accuracy_tst,
                'Random': baselines_ins[feature]['random'],
                'Majority': baselines_ins[feature]['majority']})

            break
        break
    break


# pd.DataFrame(attack_results)

      AGE     SEX SMOKER     REGION  BMI_CATEGORY  HAS_CHILDREN
626    36    male     no  northeast  2_OVERWEIGHT          True
48     60  female     no  southeast     1_HEALTHY         False
86     57  female    yes  northwest     3_OBESITY         False
1241   64    male    yes  southeast     3_OBESITY          True
923    34    male     no  northwest     3_OBESITY         False
...   ...     ...    ...        ...           ...           ...
1304   42    male    yes  northeast     1_HEALTHY          True
27     55  female     no  northwest     3_OBESITY          True
969    39  female     no  southeast     3_OBESITY          True
524    42    male    yes  southeast  2_OVERWEIGHT          True
792    22  female     no  northeast     1_HEALTHY         False

[268 rows x 6 columns]


Save _Health Insurance_ results:

In [100]:
attack_results = pd.DataFrame(attack_results)
attack_results.to_csv('results_insurance.csv', index=False)
attack_results.head()

,Dataset,Feature,Model,Variant,Split,Attack,Random,Majority
0,Insurance,SEX,XGBoost,Underfit,Train,0.504673,0.500055,0.505232
1,Insurance,SEX,XGBoost,Underfit,Test,0.507463,0.500055,0.505232
2,Insurance,SEX,XGBoost,Optimal,Train,0.518692,0.500055,0.505232
3,Insurance,SEX,XGBoost,Optimal,Test,0.477612,0.500055,0.505232
4,Insurance,SEX,XGBoost,Overfit,Train,0.539252,0.500055,0.505232


Join **POC** into `main` method:

In [131]:
def main(
        data_config: Dict[str, Any],
        model_config: Dict[str, Any],
        data_directory: str=DATA_DIRECTORY,
        random_state: int=RANDOM_STATE
    ) -> None:
    """"""

    # Models available to the experiment
    model_class = {
        'XGBoost': XGBClassifier,
        'Decision Tree': DecisionTreeClassifier,
        'Random Forest': RandomForestClassifier,
        'Multi-Layer Perceptron': MLPClassifier}

    # Iterate over each dataset
    all_results = {}
    for dataset in data_config:

        # Load dataset
        filename = data_config[dataset]['filename']
        df = pd.read_csv(os.path.join(data_directory, filename))

        # Separate features and target
        target = data_config[dataset]['target']
        X, y_raw = separate_features(df, target)
        y = encode_target(y_raw)

        # Split data into train and test
        X_trn, X_tst, y_trn, y_tst = train_test_split(
            X, y,
            test_size=0.2,
            random_state=random_state,
            stratify=y)

        # Calculate dataset  baseline
        categorical = data_config[dataset]['categorical_features']
        baselines = calculate_baselines(df, categorical)

        # Define pre-processor pipeline
        numerical = data_config[dataset]['numerical_features']
        data_preprocessor = preprocessing_pipeline(categorical, numerical)

        # Train each available model variants
        trained_variants = {}
        for model_name in model_config:
            trained_variants[model_name] = train_variants(
                model_name,
                X_trn, y_trn,
                data_preprocessor,
                model_config[model_name])

        # Iterate over each categorical feature
        attack_results = []
        for feature in categorical:
            feature_values = df[feature].unique()
            feature_baseline = baselines[feature]

            # Iterate over each model variant
            for model_name in model_config:
                model_variants = trained_variants[model_name]
                for variant_name, model_variant in model_variants.items():

                    # Attempt attack on training data
                    attack_results_trn = attribute_inference(
                        model_variant, X_trn,
                        feature, feature_values)
                    attack_accuracy_trn = attack_results_trn['is_correct'].mean()
                    attack_results.append({
                        'Dataset': dataset,
                        'Feature': feature,
                        'Model': model_name,
                        'Variant': variant_name,
                        'Split': 'Train',
                        'Attack': attack_accuracy_trn,
                        'Random': baselines[feature]['random'],
                        'Majority': baselines[feature]['majority']})

                    # Attempt attack on test data
                    attack_results_tst = attribute_inference(
                        model_variant, X_tst,
                        feature, feature_values)
                    attack_accuracy_tst = attack_results_tst['is_correct'].mean()
                    attack_results.append({
                        'Dataset': dataset,
                        'Feature': feature,
                        'Model': model_name,
                        'Variant': variant_name,
                        'Split': 'Test',
                        'Attack': attack_accuracy_tst,
                        'Random': baselines[feature]['random'],
                        'Majority': baselines[feature]['majority']})

        all_results[dataset] = pd.DataFrame(attack_results)
    return all_results


all_results = main(DATA_CONFIG, MODEL_CONFIG, DATA_DIRECTORY, RANDOM_STATE)

In [132]:
for dataset in all_results.keys():
    all_results[dataset].to_csv(f'results_{dataset}.csv', index=False)


¯\\\_(ツ)\_/¯